# Check Correlation between the Frames of the Files 

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pyhdf.SD  import *
import numpy as np
import math

### Read the Data

In [2]:
path=""
sequence=[]

for i in range(0,95):
    path="./br002_file/"+"br002_"+str(i)+".hdf"
    #print("Reading Files :",path)
    image_sequence = SD(path, SDC.READ)
    sds_obj = image_sequence.select('Data-Set-2')
    dim3 = sds_obj.get()
    frame=[]
    for i in range(0,141):
        #print("print",i)
        frame.append(dim3[:,:,i])
    

    frame=np.array(frame)
    sequence.append(frame)
    
#sequence=frame.reshape(950,128,110,1)
sequence=np.array(sequence)
data=sequence.reshape(95,141,128,110,1)
data.shape

(95, 141, 128, 110, 1)

### Cross-Correlation Calculation. 
<img src="cross-corelation.png">

In [3]:
def cor(x,y):
    sum_xy=0
    sum_x=0
    sum_y=0
 
    for i in range(128):
        for j in range(110):
            sum_xy = sum_xy + x[i,j]*y[i,j]
            sum_x = sum_x + x[i,j]*x[i,j]
            sum_y = sum_y + y[i,j]*y[i,j]
    corr = sum_xy/(np.sqrt(sum_x)*np.sqrt(sum_y))    
    return corr

#### Initialization

In [4]:
x_old = data[:,0,:,:,0]
x_new = data[:,-1,:,:,0]

print(x_old.shape)
print(x_new.shape)



(95, 128, 110)
(95, 128, 110)


In [5]:
corr_vec = np.zeros(95)
corr_vec.shape


(95,)

#### Checking for all the files - just the first and the last frames
Correlation is kept at 90%

In [6]:
for i in range(95):
    corr_vec[i] = cor(x_old[i],x_new[i])
count = np.zeros(95)
for i in range(95):
    if (corr_vec[i] > 0.90 or math.isnan(corr_vec[i])==True):
        count[i] = 1
        
sum(count)       


22.0

In [7]:
drop_index = np.where(count==1)
drop_index = np.array(drop_index)
drop_index
np.save('dropindex', drop_index)


In [8]:
drop_index = np.load('dropindex.npy')
drop_index.shape
drop_index = list(drop_index.flatten())
drop_index[:10]
data_new = data
data_new.shape


(95, 141, 128, 110, 1)

In [9]:
data_new = np.delete(data_new, drop_index, axis=0)
#data_new = data_new[:,:,:,:,np.newaxis]
data_new.shape # (73, 141, 128, 110, 1)
data = data_new
print(data.shape)#(73, 141, 128, 110, 1)
np.save('non-correlated_files', data)

(73, 141, 128, 110, 1)


In [ ]:
del data_new
del drop_index